# Whisper Transcriber — Google Colab

Jalankan app Streamlit langsung dari Colab dengan akses GPU gratis.

> **Tips:** Aktifkan GPU dulu — *Runtime → Change runtime type → T4 GPU*

Jalankan semua cell dari atas ke bawah. Link app akan muncul di cell terakhir.

In [ ]:
# ── 1. Install system dependency ─────────────────────────────
!sudo apt-get install -y ffmpeg -q

In [ ]:
# ── 2. Install Python dependencies ───────────────────────────
!pip install streamlit openai-whisper torch -q

In [ ]:
# ── 3. Download app ──────────────────────────────────────────
!wget -q https://raw.githubusercontent.com/weyennn/transcriber-audio-to-text/main/whisper_app.py
print('whisper_app.py downloaded.')

In [ ]:
# ── 4. Jalankan app + buat public URL ────────────────────────
# Menggunakan Cloudflare Tunnel (gratis, tanpa signup)

import subprocess, threading, time, re

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Jalankan Streamlit di background
def run_streamlit():
    subprocess.run(
        ['streamlit', 'run', 'whisper_app.py',
         '--server.headless', 'true',
         '--server.port', '8501',
         '--server.maxUploadSize', '500'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(4)

# Buka tunnel ke port 8501
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print('Menunggu tunnel...')
for line in tunnel.stdout:
    decoded = line.decode()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', decoded)
    if match:
        url = match.group(0)
        print(f'\n App siap! Buka di browser:')
        print(f' {url}\n')
        break